In [1]:
%matplotlib widget
import numpy as np, matplotlib.pyplot as plt, ipywidgets as widgets, imageio.v3 as iio, caiman as cm
from pathlib import Path
from pynwb import NWBHDF5IO

objc[98731]: Class AVFFrameReceiver is implemented in both /Users/albanceau/Dev/miniscope_ca1/.venv/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.63.1.102.dylib (0x101940328) and /Users/albanceau/Dev/miniscope_ca1/.venv/lib/python3.12/site-packages/av/.dylibs/libavdevice.63.1.102.dylib (0x10c5343a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[98731]: Class AVFAudioReceiver is implemented in both /Users/albanceau/Dev/miniscope_ca1/.venv/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.63.1.102.dylib (0x101940378) and /Users/albanceau/Dev/miniscope_ca1/.venv/lib/python3.12/site-packages/av/.dylibs/libavdevice.63.1.102.dylib (0x10c5343f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


In [2]:
data = Path("data/sub-F06")
ses = "sub-F06_ses-CPFE-F06-S3"

In [3]:
# processed results + behaviour; lazy, nothing is read until sliced
io = NWBHDF5IO(str(data / f"{ses}_behavior+image+ophys.nwb"), "r")
nwb = io.read()

In [4]:
# imaging frame times (s), from the raw-movie stub file
io_raw = NWBHDF5IO(str(data / f"{ses}_ophys.nwb"), "r")
t_oph = io_raw.read().acquisition["OnePhotonSeries"].timestamps[:]

In [5]:
# raw movie, downsampled x2 in space: (8935, 300, 300), ~15 s
files = sorted((data / f"{ses}_ophys").glob("*.avi"))
mov = cm.concatenate([cm.load(str(f)).resize(0.5, 0.5, 1) for f in files], axis=0)

In [6]:
# mean image, skipping the 4 black start-up frames
mean_img = mov[4:].mean(axis=0)

In [7]:
# behaviour video, infrared so the 3 channels are equal: (8998, 240, 320)
beh = iio.imread(next((data / f"{ses}_behavior+image+ophys").glob("*.wmv")), plugin="pyav")[..., 0]

In [8]:
# behaviour streams, 30 Hz from t = 0.002 s: frame k of the video is at t = 0.002 + k/30
bhv = nwb.processing["behavior"]
nose = bhv["PoseEstimation"]["nose"].data[:]
tail = bhv["PoseEstimation"]["tailbase"].data[:]
motion = bhv["BehavioralTimeSeries"]["motion_index"].data[:]
t_beh = 0.002 + np.arange(len(motion)) / 30

In [9]:
freezing = bhv["freezing"].to_dataframe()

In [10]:
# the 8 accepted cells with the highest SNR; row k of the table = column k of C
cells = nwb.processing["ophys"]["ImageSegmentation"]["PlaneSegmentation"].to_dataframe(exclude={"image_mask"})
top = cells[cells.accepted].nlargest(8, "snr").index

In [11]:
# their CNMF-E traces; row k of C = raw frame k (C has one frame less than the movie)
C = nwb.processing["ophys"]["Fluorescence"]["C"].data[:][:, top]

In [20]:
# figure layout; ioff so it is shown only in the last cell, with the controls
with plt.ioff():
    fig = plt.figure(figsize=(10, 7))
ax_raw, ax_sub, ax_beh = [fig.add_subplot(2, 3, k) for k in (1, 2, 3)]
ax_tr = fig.add_subplot(2, 1, 2)
title = fig.suptitle("")

In [13]:
# panel 1: raw frame, contrast from percentiles of a frame sample
lo, hi = np.percentile(mov[4::100], [1, 99.5])
im_raw = ax_raw.imshow(mov[4], cmap="gray", vmin=lo, vmax=hi)
ax_raw.set_title("raw");

In [14]:
# panel 2: frame minus mean image, with the 8 cells circled
lo, hi = np.percentile(mov[4::100] - mean_img, [1, 99.9])
im_sub = ax_sub.imshow(mov[4] - mean_img, cmap="gray", vmin=lo, vmax=hi)
ax_sub.scatter(cells.center_x[top] / 2, cells.center_y[top] / 2, s=80, facecolors="none", edgecolors="r")
ax_sub.set_title("raw - mean");

In [15]:
# panel 3: behaviour frame with DeepLabCut nose and tail base
im_beh = ax_beh.imshow(beh[0], cmap="gray")
dots, = ax_beh.plot([], [], "o", color="orange")
ax_beh.set_title("behaviour");

In [16]:
for ax in (ax_raw, ax_sub, ax_beh):
    ax.axis("off")

In [17]:
# bottom strip: freezing bouts shaded, motion index, normalised traces stacked
for s, e in zip(freezing.start_time, freezing.stop_time):
    ax_tr.axvspan(s, e, color="tab:blue", alpha=0.2, lw=0)
ax_tr.plot(t_beh, np.clip(motion / 600, 0, 1) - 1.2, color="k", lw=0.8)
ax_tr.plot(t_oph[:-1], C / C.max(axis=0) + np.arange(len(top)), lw=0.8)
cursor = ax_tr.axvline(0, color="r")
ax_tr.set_xlabel("time (s)")
ax_tr.set_yticks([]);

In [18]:
# redraw for behaviour frame k: imaging frame = first one at or after the same time
def update(k):
    t = t_beh[min(k, len(t_beh) - 1)]
    i = min(np.searchsorted(t_oph, t), len(mov) - 1)
    im_raw.set_data(mov[i])
    im_sub.set_data(mov[i] - mean_img)
    im_beh.set_data(beh[k])
    dots.set_data([nose[k, 0], tail[k, 0]], [nose[k, 1], tail[k, 1]])
    cursor.set_xdata([t, t])
    ax_tr.set_xlim(t - 15, t + 15)
    title.set_text(f"t = {t:.2f} s   imaging frame {i}   behaviour frame {k}")
    fig.canvas.draw_idle()

In [19]:
# controls: Play steps 3 frames every 100 ms (real time if drawing keeps up)
slider = widgets.IntSlider(min=0, max=len(beh) - 1, description="frame", layout={"width": "600px"})
play = widgets.Play(max=len(beh) - 1, step=3, interval=100)
widgets.jslink((play, "value"), (slider, "value"))
slider.observe(lambda change: update(change.new), names="value")
update(0)
widgets.VBox([widgets.HBox([play, slider]), fig.canvas])